# Certified Numerical Analysis with Rocq and proof assistant assistants

Run the setup cell once in Google Colab. It installs the workshop package, downloads the precomputed FAISS cache, and creates the retrieval/LLM clients.


## 0. Setup (no GPU required)


In [2]:
%pip install -q "integral-tp[colab] @ git+https://github.com/theostos/integral-tp.git@main"

import os
from pathlib import Path

import requests

# Temporary workshop services running on the Brev instance.
ROCQ_URL = "http://global.prd.ga.run.brev.nvidia.com:34463"
LLM_PROXY_URL = "http://global.prd.ga.run.brev.nvidia.com:61944"
WORKSHOP_TOKEN = "94ee6251fa47fe4d22cac898117aea7844da117ec581d83c"

os.environ["ROCQ_SERVER_URL"] = ROCQ_URL
os.environ["WORKSHOP_LLM_SERVER_URL"] = LLM_PROXY_URL
os.environ["WORKSHOP_LLM_SERVER_TOKEN"] = WORKSHOP_TOKEN
os.environ["WORKSHOP_EMBEDDING_SERVER_URL"] = LLM_PROXY_URL
os.environ["WORKSHOP_EMBEDDING_SERVER_TOKEN"] = WORKSHOP_TOKEN
os.environ["WORKSHOP_LLM_PROVIDER"] = "openrouter"
os.environ["OPENROUTER_MODEL"] = "z-ai/glm-5.3-flash"
os.environ["OPENROUTER_REASONING_EFFORT"] = "low"
os.environ["DOCSTRING_OPENROUTER_EMBEDDING_MODEL"] = "qwen/qwen3-embedding-4b"
os.environ["WORKSHOP_LLM_SERVER_POLL_SECONDS"] = "2.0"
os.environ["WORKSHOP_EMBEDDING_TIMEOUT_SECONDS"] = "190"

# Fail early if the temporary servers or token are no longer valid.
rocq_health = requests.get(f"{ROCQ_URL}/health", timeout=30)
rocq_health.raise_for_status()
proxy_health = requests.get(f"{LLM_PROXY_URL}/health", timeout=30)
proxy_health.raise_for_status()
queue_health = requests.get(
    f"{LLM_PROXY_URL}/queue",
    headers={"Authorization": f"Bearer {WORKSHOP_TOKEN}"},
    timeout=30,
)
queue_health.raise_for_status()
print("Rocq server: OK", ROCQ_URL)
print("LLM proxy: OK", proxy_health.json()["model"])

import workshop_api
from workshop_api import (
    LLMClient,
    RetrievalClient,
    RetrievalExplorer,
    download_retrieval_cache,
    format_retrieval_hits,
    expected_hit,
    set_hits,
    show_usage,
)

# Download the precomputed retrieval cache—not the Qwen model.
CACHE_URL = (
    "https://huggingface.co/datasets/theostos/integral-tp-retrieval-cache/"
    "resolve/main/retrieval_cache.zip"
)
cache_path = download_retrieval_cache(
    CACHE_URL, cache_dir="/content/rocq-doc-cache"
)
# The raw matrix is useful for rebuilding indexes, but FAISS search does not need it.
(Path(cache_path) / "embeddings.npy").unlink(missing_ok=True)
retriever = RetrievalClient.from_env(cache_dir=cache_path)
remote_llm = LLMClient.from_env()

# Retrieval smoke test (the query embedding is computed by the proxy).
hits = retriever.search("warmup", library="Coquelicot", k=3)
print(format_retrieval_hits(hits))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Rocq server: OK http://global.prd.ga.run.brev.nvidia.com:34463
LLM proxy: OK z-ai/glm-5.3-flash
1. point_of (definition) [score 0.296]
   library: Coquelicot
   source: Coquelicot/Hierarchy.v
   statement: Definition point_of := UniformSpace.point_of _ (UniformSpace.class M).
   docstring: A function that returns the distinguished point of a uniform space.
2. Derive (definition) [score 0.294]
   library: Coquelicot
   source: Coquelicot/Derive.v
   statement: Definition Derive (f : R -> R) (x : R) := real (Lim (fun h => (f (x+h) - f x)/h) 0).
   docstring: This definition computes the der

## How this notebook talks to Rocq

Rocq runs in a separate server. In the notebook, Python is only the interface: each call such as `add_definition`, `add_theorem`, or `run_tac` sends a Rocq command to the server and receives either new goals or an error.


The goal is not to write a complete `.v` file by hand. We interact with the prover through a higher-level layer: retrieval helps us find formal ingredients, the LLM proposes proof scripts, and Rocq checks every accepted step.

This setup builds on several related tools and collaborations: the Rocq inference server uses [`petanque`](https://github.com/rocq-community/rocq-lsp/tree/main/petanque)/[`pytanque`](https://github.com/LLM4Rocq/pytanque) and the `rocq-ml-server` from [`rocq-ml-toolbox`](https://github.com/LLM4Rocq/rocq-ml-toolbox); [`LLM4Docq`](https://github.com/LLM4Rocq/LLM4Docq), [`Pile-of-Rocq`](https://github.com/LLM4Rocq/Pile-of-rocq) dataset; and the agentic proof-assistant workflow is inspired by joint work with Guillaume Baudart, Marc Lelarge, and Jules Viennot on [`Crrrocq`](https://github.com/LLM4Rocq/crrrocq) and [`Goal2Tacq`](https://github.com/LLM4Rocq).


## Goal of the session

We start from a numerical estimate that is wrong, then ask Rocq for a certified enclosure. After that, we build an analytic closed form for the same integral.

The recurring workflow is simple: describe the formal ingredient we need, retrieve candidate lemmas or tactics, ask the LLM for the proof plumbing, and let Rocq check the result.


## 1. A numerical disagreement

We study

$$
f(x)=\operatorname{sech}(10x-2)^2
     +\operatorname{sech}(100x-40)^4
     +\operatorname{sech}(1000x-600)^6.
$$

A direct numerical integration in Mathematica gives about `0.2097360688339336`.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>🔎 Reference Mathematica call — click to reveal</strong></summary>

```Mathematica
Clear[x, f];
f[x_] := Sech[10 x - 2]^2 + Sech[100 x - 40]^4 + Sech[1000 x - 600]^6;
NIntegrate[f[x], {x, 0, 1}]
```

</details>

Sage can also be fragile here. The direct adaptive integral may miss the very narrow third peak, and with the unstable exponential definition of `sech` it may even return `nan`. A stable direct call can still give about `0.2097360688339336`, matching the Mathematica value below.

To try it yourself, open [SageMathCell](https://sagecell.sagemath.org/) and paste this code:

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>🔎 Reference Sage call — click to reveal</strong></summary>

```python
# SageMath code, to paste in https://sagecell.sagemath.org/

def sech_exp(u):
    return 2 * exp(u) / (exp(2 * u) + 1)

def sech_stable(u):
    return 1 / cosh(u)

def f_exp(x):
    return (
        (sech_exp(10 * x - 2))^2
        + (sech_exp(100 * x - 40))^4
        + (sech_exp(1000 * x - 600))^6
    )

def f_stable(x):
    return (
        (sech_stable(10 * x - 2))^2
        + (sech_stable(100 * x - 40))^4
        + (sech_stable(1000 * x - 600))^6
    )

print("unstable exponential form:")
print(numerical_integral(f_exp, 0, 1))

print("stable sech form:")
value, error = numerical_integral(f_stable, 0, 1)
print(value)
print("estimated numerical error:", error)
```
</details>

We now ask Rocq for a certified enclosure instead of trusting floating point code.


We use a first Rocq document only for the certified numerical part. Later we will create a second document for the analytic proof. This is intentional: if we reset or replay a proof while experimenting with the LLM, we do not want to trigger the expensive interval computation again.


`new_document()` creates an independent Rocq session. `add_import(lib, modules)` sends an import command to that session; here `"Coq"`, `"Coquelicot"`, and `"Interval"` are library roots, and the second argument lists modules to import.


In [3]:
doc_numerical_integral = workshop_api.new_document()

doc_numerical_integral.add_import("Coq", "Reals Lra Psatz")
doc_numerical_integral.add_import("Coquelicot", "Coquelicot")
doc_numerical_integral.add_import("Interval", "Tactic Plot")


{'ok': True,
 'kind': 'element',
 'command': 'From Interval Require Import Tactic Plot.',
 'feedback': ['Declaring a scope implicitly is deprecated; use in advance an explicit\n"Declare Scope bigQ_scope.".\n[undeclared-scope,deprecated-since-8.10,deprecated,default]'],
 'refreshed_open_lemmas': []}

`add_definition(command)` sends one complete Rocq `Definition`. The string must include the name, arguments, body, and final period.


In [4]:
doc_numerical_integral.add_definition("""Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).""")

doc_numerical_integral.add_definition("""Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.""")

doc_numerical_integral.add_definition("Definition I : R := RInt f 0 1.")


{'ok': True,
 'kind': 'element',
 'command': 'Definition I : R := RInt f 0 1.',
 'feedback': ['I is defined'],
 'refreshed_open_lemmas': []}

The command below asks Interval to compute a certified decimal enclosure. It is the only heavy numerical cell in the notebook.


`execute(command)` sends a standalone Rocq command. Here we use it only for the heavy interval computation.


In [5]:
result = doc_numerical_integral.execute("""Do integral
  ltac:(let J := eval cbv [I f sech] in I in exact J)
  with (i_prec 25, i_degree 3, i_fuel 300,
        i_width (-15), i_decimal).""")

print(result['feedback'][0])

(0.21078887 <=
 RInt
   (fun x : R =>
    (2 * exp (10 * x - 2) / (exp (2 * (10 * x - 2)) + 1)) ^ 2 +
    (2 * exp (100 * x - 40) / (exp (2 * (100 * x - 40)) + 1)) ^ 4 +
    (2 * exp (1000 * x - 600) / (exp (2 * (1000 * x - 600)) + 1)) ^ 6) 0 1 <=
 0.21081871)


On the reference file this prints:

```text
0.21078887 <= I <= 0.21081871
```

This is incompatible with the value printed by Mathematica/Sage. We can package the statement as a theorem. This is the only proof in the session that we run manually from start to finish.


`add_theorem(header)` opens a proof session. `run_tac(tactic)` sends one tactic command to the current proof state, and `qed()` closes the theorem once no goals remain.


In [6]:
I_digits = doc_numerical_integral.ensure_theorem("""Theorem I_first_4_decimal_digits :
  Rabs (I - 0.2108) <= 1e-4.""")

I_digits.run_tac("unfold I, f, sech.")
I_digits.run_tac("integral with (i_prec 25, i_degree 3, i_fuel 300).")
I_digits.qed()


{'ok': True,
 'lemma': 'I_first_4_decimal_digits',
 'feedback': ['I_first_4_decimal_digits is defined'],
 'refreshed_open_lemmas': []}

## 2. Rocq objects we will manipulate

A few words will appear throughout the notebook:

- a **goal** is what remains to prove;
- a **theorem** or **lemma** is a named statement, reusable after it is proved;
- a **tactic** is a Rocq command that transforms the current goals;
- a **retrieval hit** is a library fact or tactic that we may give to the LLM.

In practice, we will collect a small context of useful hits, then ask the LLM to assemble a proof script from the current proof state.


## 3. Analytic document

We now start a fresh Rocq document. It repeats the definitions, but it does not run the interval computation.


In [7]:
doc_analytic_integral = workshop_api.new_document()

doc_analytic_integral.add_import("Coq", "Reals Lra Psatz")
doc_analytic_integral.add_import("Coquelicot", "Coquelicot")
doc_analytic_integral.add_import("Interval", "Tactic Plot")


{'ok': True,
 'kind': 'element',
 'command': 'From Interval Require Import Tactic Plot.',
 'feedback': ['Declaring a scope implicitly is deprecated; use in advance an explicit\n"Declare Scope bigQ_scope.".\n[undeclared-scope,deprecated-since-8.10,deprecated,default]'],
 'refreshed_open_lemmas': []}

In [8]:
doc_analytic_integral.add_definition("""Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).""")

doc_analytic_integral.add_definition("""Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.""")

doc_analytic_integral.add_definition("Definition I : R := RInt f 0 1.")


{'ok': True,
 'kind': 'element',
 'command': 'Definition I : R := RInt f 0 1.',
 'feedback': ['I is defined'],
 'refreshed_open_lemmas': []}

The analytic candidate uses the exponential presentation of the hyperbolic tangent:

$$
\tanh(u)=\frac{e^{2u}-1}{e^{2u}+1}.
$$

We use the Rocq name `tanh_exp` because Stdlib already defines `tanh` through `sinh` and `cosh`.


For the three powers of `sech`, we use these one-variable candidates:

$$
A_2(u)=\tanh(u),\qquad
A_4(u)=\tanh(u)-\frac{1}{3}\tanh(u)^3,
$$

$$
A_6(u)=\tanh(u)-\frac{2}{3}\tanh(u)^3+\frac{1}{5}\tanh(u)^5.
$$


Then we rescale them to match the three bumps of the integrand:

$$
F_2(x)=\frac{A_2(10x-2)}{10},\qquad
F_4(x)=\frac{A_4(100x-40)}{100},\qquad
F_6(x)=\frac{A_6(1000x-600)}{1000}.
$$

So `F2`, `F4`, and `F6` correspond to the square, fourth-power, and sixth-power terms respectively. The proposed closed form is

$$
F(1)-F(0)\quad\text{where}\quad F=F_2+F_4+F_6.
$$


In [9]:
doc_analytic_integral.add_definition("""Definition tanh_exp (u : R) : R :=
  (exp (2 * u) - 1) / (exp (2 * u) + 1).""")

doc_analytic_integral.add_definition("""Definition A2 (u : R) : R :=
  tanh_exp u.""")

doc_analytic_integral.add_definition("""Definition A4 (u : R) : R :=
  tanh_exp u - (/ 3) * (tanh_exp u)^3.""")

doc_analytic_integral.add_definition("""Definition A6 (u : R) : R :=
  tanh_exp u - (2 / 3) * (tanh_exp u)^3 + (/ 5) * (tanh_exp u)^5.""")


{'ok': True,
 'kind': 'element',
 'command': 'Definition A6 (u : R) : R :=\n  tanh_exp u - (2 / 3) * (tanh_exp u)^3 + (/ 5) * (tanh_exp u)^5.',
 'feedback': ['A6 is defined'],
 'refreshed_open_lemmas': []}

Now we define the rescaled functions `F2`, `F4`, `F6`, their sum `F`, and the candidate value `I_closed_form`.


In [10]:
doc_analytic_integral.add_definition("""Definition F2 (x : R) : R :=
  A2 (10 * x - 2) / 10.""")

doc_analytic_integral.add_definition("""Definition F4 (x : R) : R :=
  A4 (100 * x - 40) / 100.""")

doc_analytic_integral.add_definition("""Definition F6 (x : R) : R :=
  A6 (1000 * x - 600) / 1000.""")

doc_analytic_integral.add_definition("""Definition F (x : R) : R :=
  F2 x + F4 x + F6 x.""")

doc_analytic_integral.add_definition("""Definition I_closed_form : R :=
  F 1 - F 0.""")


{'ok': True,
 'kind': 'element',
 'command': 'Definition I_closed_form : R :=\n  F 1 - F 0.',
 'feedback': ['I_closed_form is defined'],
 'refreshed_open_lemmas': []}

The formal work now has two phases:

1. prove that `F2`, `F4`, and `F6` differentiate to the three terms of `f`;
2. combine those facts and apply the fundamental theorem of calculus.


## 4. Formalizing `F2_derivative`

First we need to express the informal statement "the derivative of `F2` is `sech(10*x - 2)^2`" in Coquelicot's language.

Search for the predicate used to say that a real function has a derivative at a point.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint — click to reveal</strong></summary>

Try a query like: `real function has derivative at a point`.

</details>


## Retrieval in one minute

The retrieval system is a semantic search engine over Rocq documentation. Each library item was converted offline into a vector, called an **embedding**, by `Qwen/Qwen3-Embedding-4B`. For each search, the matching OpenRouter model `qwen/qwen3-embedding-4b` converts only your query into the same vector space through the workshop proxy; the local FAISS index then finds nearby library items.


The notebook compares the query vector with all documentation vectors using cosine similarity: after normalization, this is just a dot product. High cosine means that the texts point in a similar direction, so we show the nearest Rocq definitions, lemmas, or tactics.


The search is only a suggestion mechanism. Rocq does not trust the retriever or the LLM: only the final proof script checked by the Rocq kernel matters.


`RetrievalExplorer(retriever, selected_hits=..., default_query=..., default_library=..., default_kind=..., default_k=...)` opens a small search UI. The explorer appends chosen hits to `selected_hits`; the other arguments only initialize the search fields.


In [11]:
# This list is the context that we curate during retrieval.
# RetrievalExplorer writes into it, and we can also add already proved objects.
selected_hits = []

selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="derivative of a function at a point",
    default_kind="definition",
    default_library="Coquelicot",
    default_k=8,
)
retrieval_explorer.display()


Based on the search results, did you find a definition that can express the derivative of `F2`?

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hit — click to reveal</strong></summary>

Name: `is_derive`

Signature:

```coq
is_derive : (R -> R) -> R -> R -> Prop
```

</details>

Now fill the following cell with the formal statement. The declaration is rerunnable: if Rocq accepted a statement but you later notice it is wrong, edit it and rerun the cell while the theorem is still open. Once a theorem has been closed with `Qed`, Rocq correctly prevents redefining it in the same document.


In [15]:
f2_derivative = doc_analytic_integral.ensure_theorem("""Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech(10*x - 2))^2).""")

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected answer — click to reveal</strong></summary>

```coq
Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech (10 * x - 2)) ^ 2).
```

</details>


To prove this, we need to show two things:

- `F2` is differentiable at `x`;
- the derivative computed from the definition of `F2` is equal to the right-hand side.

So we are looking for a tactic that helps with differentiability and computes derivatives of real functions.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint — click to reveal</strong></summary>

Try a query like: `a tactic to obtain automatically the derivative`.

</details>

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hit — click to reveal</strong></summary>

Name: `auto_derive`

Signature:

```coq
Ltac auto_derive := ...
```

It is a Coquelicot tactic for derivative and differentiability goals.

</details>


In [16]:
selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="Ltac",
    default_k=8,
)
retrieval_explorer.display()


Now we run only the first proof steps by hand. This is useful: `unfold` exposes the definitions, and the derivative tactic exposes the side conditions that are really needed.


In [17]:
f2_derivative.run_tac("unfold F2, A2, sech, tanh_exp.")
f2_derivative.run_tac("auto_derive.")
print(f2_derivative)


F2_derivative: 2 goal(s)

Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech(10*x - 2))^2).

Goals:
x  : R
|-exp (2 * (10 * x + - (2))) + 1 <> 0

x  : R
|-(2 * (10 * 1) * exp (2 * (10 * x + - (2))) *
 / (exp (2 * (10 * x + - (2))) + 1) +
 (exp (2 * (10 * x + - (2))) + - (1)) *
 (- (2 * (10 * 1) * exp (2 * (10 * x + - (2)))) *
  / ((exp (2 * (10 * x + - (2))) + 1) * (exp (2 * (10 * x + - (2))) + 1)))) *
/ 10 = (2 * exp (10 * x - 2) / (exp (2 * (10 * x - 2)) + 1)) ^ 2


`checkpoint("name")` records the current proof state. Later, `reverse("name")` rolls the proof back to that state; this is how the three LLM strategies start from the same goal.


In [18]:
# here we do a checkpoint so that later we will be able to
# reset the proof to this more advance point
f2_derivative.checkpoint("after_auto_derive")

'after_auto_derive'

Inspect the goals. One side condition says that an expression of the form `exp u + 1` is not zero.

This denominator appears several times, so we prove it once as a separate lemma.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint — click to reveal</strong></summary>

Search (in different queries) for facts saying:
- `exp` is positive
- a sum of positive real numbers is positive
- a positive real number is not zero.

</details>

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hits — click to reveal</strong></summary>

```coq
exp_pos : forall x : R, 0 < exp x
Rplus_lt_0_compat : forall r1 r2 : R, 0 < r1 -> 0 < r2 -> 0 < r1 + r2
Rgt_not_eq : forall r1 r2 : R, r1 > r2 -> r1 <> r2
```

</details>


In [19]:
selected_hits.clear()

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Stdlib",
    default_kind="theorem",
    default_k=8,
)
retrieval_explorer.display()


`set_hits(selected_hits, ...)` clears the selected context and reloads the listed ingredients. It is the fallback path when the explorer was hard to use.

`remote_llm.prove(theorem, selected_hits=..., extra_context=..., tools=..., max_tool_calls=...)` asks the LLM for proof steps and lets Rocq check them. `selected_hits` is the formal context, `extra_context` is informal mathematical guidance, and `tools` enables the agentic loop with `run_tac` and `reverse`.


In [20]:
set_hits(
    selected_hits,
    expected_hit(retriever, "Rgt_not_eq", library="Stdlib", kind="theorem"),
    expected_hit(retriever, "Rplus_lt_0_compat", library="Stdlib", kind="theorem"),
    expected_hit(retriever, "exp_pos", library="Stdlib", kind="theorem"),
)


['Rgt_not_eq', 'Rplus_lt_0_compat', 'exp_pos']


[{'content': 'Lemma Rgt_not_eq : forall r1 r2, r1 > r2 -> r1 <> r2.',
  'data_uid': 'Rgt_not_eq',
  'depth': 0,
  'docstring': 'This lemma states that if one real number is strictly greater than another, then the two numbers are not equal.',
  'env_id': 'coq-coquelicot',
  'kind': 'theorem',
  'library': 'Stdlib',
  'name': 'Rgt_not_eq',
  'preorder_index': 45,
  'range': [3830, 3883],
  'source': 'Corelib/Reals/RIneq.v',
  'source_id': 'b2ff80bfe21d2b23e345bee356705d7fc4b9a1a487182c8797b7d2ce5bd6829e',
  'uid': 'fe8c893d003ebc606ec07a5f1b41551b39ba265a',
  'score': 1.0},
 {'content': 'Lemma Rplus_lt_0_compat : forall r1 r2, 0 < r1 -> 0 < r2 -> 0 < r1 + r2.',
  'data_uid': 'Rplus_lt_0_compat',
  'depth': 0,
  'docstring': 'This lemma states that the sum of two strictly positive real numbers is strictly positive.',
  'env_id': 'coq-coquelicot',
  'kind': 'theorem',
  'library': 'Stdlib',
  'name': 'Rplus_lt_0_compat',
  'preorder_index': 1001,
  'range': [36373, 36445],
  'source': 'Cor

In [21]:
sech_denominator_nonzero = doc_analytic_integral.ensure_theorem("""Lemma sech_denominator_nonzero (u : R) :
  exp u + 1 <> 0.""")


In [22]:
if sech_denominator_nonzero.completed:
    print("sech_denominator_nonzero is already closed.")
else:
    result = remote_llm.prove(
        sech_denominator_nonzero,
        selected_hits=selected_hits,
        extra_context=(
            "denominator is nonzero because it is strictly "
            "positive: the exponential is positive, and adding 1 preserves "
            "strict positivity."
        ),
        verbose=True,
        tools={
            "run_tac": sech_denominator_nonzero.run_tac,
            "reverse": sech_denominator_nonzero.reverse,
        },
        max_tool_calls=20,
    )
    print(result)


[llm] requesting tool...
[llm:queue] running | queued=0 | running=2 | attempts=1
[llm:queue] succeeded | queued=0 | running=1 | attempts=1
[llm] response received (99 chars) | input=2417, output=36, cost=$0.000190 | queue wait=0.0s, attempts=1, retries=0
[tool] finish: {'action': 'finish', 'script': 'apply Rgt_not_eq.\napply Rplus_lt_0_compat.\napply exp_pos.\nlra.'}
[script]
apply Rgt_not_eq.
apply Rplus_lt_0_compat.
apply exp_pos.
lra.
[rocq] apply Rgt_not_eq.
[rocq] ok, goals=1
[rocq] apply Rplus_lt_0_compat.
[rocq] ok, goals=2
[rocq] apply exp_pos.
[rocq] ok, goals=1
[rocq] [cleanup] lra.
[rocq] ok, goals=0
[rocq] Qed ok
ProofResult(ok)

usage: input=2417, output=36, cost=$0.000190

script:
apply Rgt_not_eq.
apply Rplus_lt_0_compat.
apply exp_pos.
lra.


> 🛟 **Executable fallback below.** The next code cell is intentionally folded. In JupyterLab, click its `…` control (or the blue input gutter at the left) to reveal it, then run the cell.


In [ ]:
#@title Fallback: prove `sech_denominator_nonzero` { display-mode: "form" }
if sech_denominator_nonzero.completed:
    print("sech_denominator_nonzero is already closed.")
else:
    sech_denominator_nonzero.reset()
    _outputs = sech_denominator_nonzero.run_script([
        "apply Rgt_not_eq.",
        "apply Rplus_lt_0_compat.",
        "apply exp_pos.",
        "lra.",
    ])
    for _out in _outputs:
        if not _out.get("ok"):
            raise RuntimeError(_out)
    _qed = sech_denominator_nonzero.qed()
    if not _qed.get("ok"):
        raise RuntimeError(_qed)
    print(_qed)


The lemma is now a local result we proved during the session. We add it to the selected context so the LLM can use it exactly like a library fact.

There is one more technical ingredient in the derivative proofs. After `auto_derive`, the equality goal contains expressions such as `exp (2 * u)`, while `sech` contains `exp u`. We need the theorem saying that the exponential of a sum is a product.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint — click to reveal</strong></summary>

Try a query like: `exponential of a sum`.

</details>

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hit — click to reveal</strong></summary>

Name: `exp_plus`

Signature:

```coq
exp_plus : forall x y : R, exp (x + y) = exp x * exp y
```

</details>


In [23]:
selected_hits.clear()
selected_hits.append(sech_denominator_nonzero.as_retrieval_hit())

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Stdlib",
    default_kind="theorem",
    default_k=8,
)
retrieval_explorer.display()


After adding `exp_plus`, save this small technical context. It will be reused for `F2`, `F4`, and `F6`.


Safety net: this cell loads the expected technical context. It does not replace the retrieval exercise; it just restores the ingredients needed by the proof cells below.


In [24]:
set_hits(
    selected_hits,
    sech_denominator_nonzero.as_retrieval_hit(),
    expected_hit(retriever, "exp_plus", library="Stdlib", kind="theorem"),
)
technical_derivative_hits = list(selected_hits)
print(format_retrieval_hits(technical_derivative_hits))


['sech_denominator_nonzero', 'exp_plus']
1. sech_denominator_nonzero (local_theorem)
   library: Current document
   source: current Rocq session
   statement: Lemma sech_denominator_nonzero (u : R) : exp u + 1 <> 0.
   docstring: Lemma sech_denominator_nonzero (u : R) : exp u + 1 <> 0. Proof. apply Rgt_not_eq. apply Rplus_lt_0_compat. apply exp_pos. lra. Qed.
2. exp_plus (theorem) [score 1.000]
   library: Stdlib
   source: Corelib/Reals/Exp_prop.v
   statement: Lemma exp_plus : forall x y:R, exp (x + y) = exp x * exp y.
   docstring: This lemma states that for any two real numbers, the exponential function evaluated at the sum of the two real numbers equals the product of the exponential function evaluated at each real number separately.


## 5. Three LLM strategies on `F2_derivative`

Here the LLM is an untrusted text model: it proposes Rocq commands, but Rocq decides whether they are valid. We compare three ways to ask it to finish the same proof state.

Before each experiment, we roll back to the checkpoint just after `auto_derive`. The selected context should contain `sech_denominator_nonzero` and `exp_plus`. The printed result includes input tokens, output tokens, and estimated dollar cost.


Strategy A is a single non-agentic call: the model receives the proof state and selected context, then returns one complete proof script. It is the cheapest when it works, but it gets no Rocq feedback before committing to the script.


In [25]:
set_hits(selected_hits, *technical_derivative_hits)


['sech_denominator_nonzero', 'exp_plus']


[{'uid': 'local:sech_denominator_nonzero',
  'name': 'sech_denominator_nonzero',
  'kind': 'local_theorem',
  'library': 'Current document',
  'source': 'current Rocq session',
  'statement': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.',
  'docstring': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.\nProof.\n  apply Rgt_not_eq.\n  apply Rplus_lt_0_compat.\n  apply exp_pos.\n  lra.\nQed.\n',
  'content': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.\nProof.\n  apply Rgt_not_eq.\n  apply Rplus_lt_0_compat.\n  apply exp_pos.\n  lra.\nQed.\n'},
 {'content': 'Lemma exp_plus : forall x y:R, exp (x + y) = exp x * exp y.',
  'data_uid': 'exp_plus',
  'depth': 0,
  'docstring': 'This lemma states that for any two real numbers, the exponential function evaluated at the sum of the two real numbers equals the product of the exponential function evaluated at each real number separately.',
  'env_id': 'coq-coquelicot',
  'kind': 'theorem',
  'library': 'S

In [26]:
f2_max_attempts = 3
result_direct = None
direct_results = []

for attempt_id in range(1, f2_max_attempts + 1):
    f2_derivative.reverse("after_auto_derive")
    print(f"Direct attempt {attempt_id}/{f2_max_attempts}")
    result_direct = remote_llm.prove(
        f2_derivative,
        selected_hits=selected_hits,
        verbose=True,
        close=False,
    )
    direct_results.append(result_direct)
    print(result_direct)
    if result_direct.ok:
        break

show_usage("Strategy A total", direct_results)
if result_direct is None or not result_direct.ok:
    print("Strategy A did not close the proof; continue to compare feedback and tool calling.")
f2_derivative.reverse("after_auto_derive")


Direct attempt 1/3
[llm] requesting proof...
[llm:queue] running | queued=0 | running=2 | attempts=1
[llm:queue] succeeded | queued=0 | running=1 | attempts=1
[llm] response received (361 chars) | input=1839, output=2687, cost=$0.000810 | queue wait=0.0s, attempts=1, retries=0
[script]
apply (sech_denominator_nonzero (2 * (10 * x - 2))).
assert (Eh : 0 < exp (10 * x - 2)) by apply exp_pos.
assert (H : exp (2 * (10 * x - 2)) = exp (10 * x - 2) ^ 2).
{ replace (2 * (10 * x - 2)) with ((10 * x - 2) + (10 * x - 2)) by ring.
  rewrite exp_plus. ring. }
rewrite H.
field; try nra.
apply Rgt_not_eq; apply Rplus_lt_0_compat; [ apply exp_pos | lra ].
[rocq] apply (sech_denominator_nonzero (2 * (10 * x - 2))).
[rocq] ok, goals=1
[rocq] assert (Eh : 0 < exp (10 * x - 2)) by apply exp_pos.
[rocq] ok, goals=1
[rocq] assert (H : exp (2 * (10 * x - 2)) = exp (10 * x - 2) ^ 2).
[rocq] ok, goals=2
[rocq] { replace (2 * (10 * x - 2)) with ((10 * x - 2) + (10 * x - 2)) by ring.
[rocq] ok, goals=1
[rocq] r

{'ok': True,
 'lemma': 'F2_derivative',
 'latest_state_index': 2,
 'goals': ['x  : R\n|-exp (2 * (10 * x + - (2))) + 1 <> 0',
  'x  : R\n|-(2 * (10 * 1) * exp (2 * (10 * x + - (2))) *\n / (exp (2 * (10 * x + - (2))) + 1) +\n (exp (2 * (10 * x + - (2))) + - (1)) *\n (- (2 * (10 * 1) * exp (2 * (10 * x + - (2)))) *\n  / ((exp (2 * (10 * x + - (2))) + 1) * (exp (2 * (10 * x + - (2))) + 1)))) *\n/ 10 = (2 * exp (10 * x - 2) / (exp (2 * (10 * x - 2)) + 1)) ^ 2']}

Strategy B is still non-agentic, but with feedback. If the first complete script fails, we call the model again with the previous script and Rocq error. This usually costs more than A, but the error message often makes the next attempt much better.


In [27]:
set_hits(selected_hits, *technical_derivative_hits)


['sech_denominator_nonzero', 'exp_plus']


[{'uid': 'local:sech_denominator_nonzero',
  'name': 'sech_denominator_nonzero',
  'kind': 'local_theorem',
  'library': 'Current document',
  'source': 'current Rocq session',
  'statement': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.',
  'docstring': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.\nProof.\n  apply Rgt_not_eq.\n  apply Rplus_lt_0_compat.\n  apply exp_pos.\n  lra.\nQed.\n',
  'content': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.\nProof.\n  apply Rgt_not_eq.\n  apply Rplus_lt_0_compat.\n  apply exp_pos.\n  lra.\nQed.\n'},
 {'content': 'Lemma exp_plus : forall x y:R, exp (x + y) = exp x * exp y.',
  'data_uid': 'exp_plus',
  'depth': 0,
  'docstring': 'This lemma states that for any two real numbers, the exponential function evaluated at the sum of the two real numbers equals the product of the exponential function evaluated at each real number separately.',
  'env_id': 'coq-coquelicot',
  'kind': 'theorem',
  'library': 'S

In [53]:
f2_derivative.reverse("after_auto_derive")


def result_ok(result):
    if isinstance(result, dict):
        return bool(result.get("ok"))
    return bool(getattr(result, "ok", False))


def result_part(result, name):
    if isinstance(result, dict):
        return result.get(name, "")
    return getattr(result, name, "")


feedback_context = ""
result_feedback = None
feedback_results = []

for round_id in range(3):
    f2_derivative.reverse("after_auto_derive")
    print(f"Feedback attempt {round_id}/2")
    result_feedback = remote_llm.prove(
        f2_derivative,
        selected_hits=selected_hits,
        extra_context=feedback_context,
        verbose=True,
        close=False,
    )
    feedback_results.append(result_feedback)
    print(result_feedback)

    if result_ok(result_feedback):
        break

    feedback_context += f"""
Previous attempt #{round_id + 1}:
{result_part(result_feedback, "script")}

Rocq feedback:
{result_part(result_feedback, "error")}
"""

show_usage("Strategy B total", feedback_results)
if result_feedback is None or not result_ok(result_feedback):
    print("Strategy B did not close the proof; continue to the tool-calling strategy.")
f2_derivative.reverse("after_auto_derive")


ValueError: Cannot reverse completed theorem `F2_derivative`.

Strategy C is agentic. Instead of returning a full script immediately, the model can call `run_tac` to try one tactic, inspect the new Rocq goals, and call `reverse` to roll back. It is usually the most reliable, but it can be the most expensive because each tool step may require another LLM request.


In [29]:
set_hits(selected_hits, *technical_derivative_hits)


['sech_denominator_nonzero', 'exp_plus']


[{'uid': 'local:sech_denominator_nonzero',
  'name': 'sech_denominator_nonzero',
  'kind': 'local_theorem',
  'library': 'Current document',
  'source': 'current Rocq session',
  'statement': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.',
  'docstring': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.\nProof.\n  apply Rgt_not_eq.\n  apply Rplus_lt_0_compat.\n  apply exp_pos.\n  lra.\nQed.\n',
  'content': 'Lemma sech_denominator_nonzero (u : R) :\n  exp u + 1 <> 0.\nProof.\n  apply Rgt_not_eq.\n  apply Rplus_lt_0_compat.\n  apply exp_pos.\n  lra.\nQed.\n'},
 {'content': 'Lemma exp_plus : forall x y:R, exp (x + y) = exp x * exp y.',
  'data_uid': 'exp_plus',
  'depth': 0,
  'docstring': 'This lemma states that for any two real numbers, the exponential function evaluated at the sum of the two real numbers equals the product of the exponential function evaluated at each real number separately.',
  'env_id': 'coq-coquelicot',
  'kind': 'theorem',
  'library': 'S

In [30]:
f2_derivative.reverse("after_auto_derive")

f2_agentic_context = """
For the exponential identity, first express each doubled affine argument as a
sum of two identical copies of that same affine argument.
"""

result_agentic = remote_llm.prove(
    f2_derivative,
    selected_hits=selected_hits,
    extra_context=f2_agentic_context,
    verbose=True,
    tools={
        "run_tac": f2_derivative.run_tac,
        "reverse": f2_derivative.reverse,
    },
    max_tool_calls=20,
)
print(result_agentic)
show_usage("Strategy C total", result_agentic)


[llm] requesting tool...
[llm:queue] running | queued=0 | running=3 | attempts=1
[llm:queue] succeeded | queued=0 | running=2 | attempts=1
[llm] response received (188 chars) | input=2602, output=125, cost=$0.000453 | queue wait=0.0s, attempts=1, retries=0
[tool] run_tac: {'action': 'run_tac', 'tactic': 'apply (sech_denominator_nonzero (2 * (10 * x - 2))).', 'allow_new_goals': True, 'reason': 'Solves first goal; second goal is the main derivative identity'}
[rocq] apply (sech_denominator_nonzero (2 * (10 * x - 2))).
[rocq] ok, goals=1
[llm] requesting tool...
[llm:queue] running | queued=0 | running=3 | attempts=1
[llm:queue] succeeded | queued=0 | running=3 | attempts=1
[llm] response received (197 chars) | input=2934, output=1428, cost=$0.001154 | queue wait=0.0s, attempts=1, retries=0
[tool] run_tac: {'action': 'run_tac', 'tactic': 'replace (2 * (10 * x + - (2))) with ((10 * x - 2) + (10 * x - 2)) by ring.', 'allow_new_goals': False, 'reason': 'align exponential argument with exp_pl

LLMUsage(model='z-ai/glm-5.3-flash', input_tokens=12562, output_tokens=3809, total_tokens=16371, cache_tokens=1216, uncached_input_tokens=11346, input_cost_usd=0.00085095, cached_input_cost_usd=1.8239999999999998e-05, output_cost_usd=0.00095225, total_cost_usd=0.00341684)

> 🛟 **Executable fallback below.** The next code cell is intentionally folded. In JupyterLab, click its `…` control (or the blue input gutter at the left) to reveal it, then run the cell.


In [ ]:
#@title Fallback: prove `F2_derivative` { display-mode: "form" }
if f2_derivative.completed:
    print("F2_derivative is already closed.")
else:
    f2_derivative.reset()
    _outputs = f2_derivative.run_script([
        "unfold F2, A2, sech, tanh_exp.",
        "auto_derive.",
        "apply sech_denominator_nonzero.",
        "simpl.",
        "replace (10 * x + - (2)) with (10 * x - 2) by ring.",
        "replace (2 * (10 * x - 2)) with ((10 * x - 2) + (10 * x - 2)) by ring.",
        "rewrite exp_plus.",
        "field.",
        "nra.",
    ])
    for _out in _outputs:
        if not _out.get("ok"):
            raise RuntimeError(_out)
    _qed = f2_derivative.qed()
    if not _qed.get("ok"):
        raise RuntimeError(_qed)
    print(_qed)


The costs should not be read as a leaderboard independent of the proof. A can be very cheap but brittle; B pays for retries; C pays for interaction with Rocq. In this TP, the interesting point is that feedback and tool use can turn a weak first attempt into a checked proof.


After one strategy succeeds, the proof is available as a local Rocq object. We will use it in two different ways: later as a formal lemma for the derivative of `F`, and now as an example passed through `extra_context` for similar derivative proofs.


In [31]:
# Keep the selected formal context for derivative proofs minimal.
# The F2 proof itself will be passed as extra_context when it is useful as an example.
set_hits(selected_hits, *technical_derivative_hits)
print(format_retrieval_hits(selected_hits))


['sech_denominator_nonzero', 'exp_plus']
1. sech_denominator_nonzero (local_theorem)
   library: Current document
   source: current Rocq session
   statement: Lemma sech_denominator_nonzero (u : R) : exp u + 1 <> 0.
   docstring: Lemma sech_denominator_nonzero (u : R) : exp u + 1 <> 0. Proof. apply Rgt_not_eq. apply Rplus_lt_0_compat. apply exp_pos. lra. Qed.
2. exp_plus (theorem) [score 1.000]
   library: Stdlib
   source: Corelib/Reals/Exp_prop.v
   statement: Lemma exp_plus : forall x y:R, exp (x + y) = exp x * exp y.
   docstring: This lemma states that for any two real numbers, the exponential function evaluated at the sum of the two real numbers equals the product of the exponential function evaluated at each real number separately.


## 6. `F4_derivative`

The fourth-power proof has the same shape as `F2_derivative`. The selected formal ingredients are still only the denominator lemma and `exp_plus`.

The completed `F2_derivative` proof is useful, but not as a formal ingredient here: we pass it separately through `extra_context` as a nearby successful proof for the model to imitate.


In [32]:
f4_derivative = doc_analytic_integral.ensure_theorem("""Lemma F4_derivative (x : R) :
  is_derive F4 x ((sech (100 * x - 40)) ^ 4).""")

selected_hits.clear()
selected_hits.extend(technical_derivative_hits)

The cell below uses the agentic strategy again. The selected context contains the reusable denominator lemma and the exponential addition theorem. The successful `F2_derivative` proof is passed separately as `extra_context`, as an example of a nearby proof shape.


In [33]:
set_hits(
    selected_hits,
    expected_hit(retriever, "auto_derive", library="Coquelicot", kind="ltac"),
    *technical_derivative_hits,
)


['auto_derive', 'sech_denominator_nonzero', 'exp_plus']


[{'content': 'Ltac auto_derive :=\n  match goal with\n  | |- is_derive ?f ?v ?l =>\n    auto_derive_fun f ;\n    let H := fresh "H" in\n    intro H ;\n    refine (@eq_ind R _ (is_derive f v) (H v _) l _) ;\n    clear H\n  | |- ex_derive ?f ?v =>\n    eexists ;\n    auto_derive_fun f ;\n    let H := fresh "H" in\n    intro H ;\n    apply (H v) ;\n    clear H\n  | |- derivable_pt_lim ?f ?v ?l =>\n    apply is_derive_Reals ;\n    auto_derive\n  | |- derivable_pt ?f ?v =>\n    apply ex_derive_Reals_0 ;\n    auto_derive\n  end.',
  'data_uid': 'auto_derive',
  'depth': 0,
  'docstring': 'A tactic that automatically establishes the differentiability of real-valued functions or computes their derivatives at specified points by processing goals concerning derivative existence, derivative values, or pointwise derivability. The tactic applies a reflective algorithm that symbolically differentiates expressions and reduces proof obligations to verification of domain conditions including non-zero d

In [34]:
f4_example_context = """
Here is a closely related successful proof. Use it as an example of structure,
while adapting the details to the current goal.

```coq
Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech (10 * x - 2)) ^ 2).
Proof.
  unfold F2, A2, sech, tanh_exp.
  auto_derive.
  apply sech_denominator_nonzero.
  simpl.
  replace (10 * x + - (2)) with (10 * x - 2) by ring.
  replace (2 * (10 * x - 2)) with ((10 * x - 2) + (10 * x - 2)) by ring.
  rewrite exp_plus.
  field.
  nra.
Qed.
```
"""

result = remote_llm.prove(
    f4_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f4_derivative.run_tac,
        "reverse": f4_derivative.reverse,
    },
    max_tool_calls=20,
    extra_context=f4_example_context,
)
print(result)
show_usage("F4_derivative", result)


[llm] requesting tool...
[llm:queue] running | queued=0 | running=6 | attempts=1
[llm:queue] succeeded | queued=0 | running=4 | attempts=1
[llm] response received (326 chars) | input=2983, output=133, cost=$0.000514 | queue wait=0.0s, attempts=1, retries=0
[tool] finish: {'action': 'finish', 'script': 'unfold F4, A4, sech, tanh_exp.\nauto_derive.\napply (sech_denominator_nonzero (2 * (100 * x - 40))).\nreplace (2 * (100 * x + - (40))) with ((100 * x - 40) + (100 * x - 40)) by ring.\nreplace (2 * (100 * x - 40)) with ((100 * x - 40) + (100 * x - 40)) by ring.\nrewrite exp_plus.\nfield; nra.'}
[script]
unfold F4, A4, sech, tanh_exp.
auto_derive.
apply (sech_denominator_nonzero (2 * (100 * x - 40))).
replace (2 * (100 * x + - (40))) with ((100 * x - 40) + (100 * x - 40)) by ring.
replace (2 * (100 * x - 40)) with ((100 * x - 40) + (100 * x - 40)) by ring.
rewrite exp_plus.
field; nra.
[rocq] unfold F4, A4, sech, tanh_exp.
[rocq] ok, goals=1
[rocq] auto_derive.
[rocq] ok, goals=2
[rocq] ap

LLMUsage(model='z-ai/glm-5.3-flash', input_tokens=50034, output_tokens=2114, total_tokens=52148, cache_tokens=12864, uncached_input_tokens=37170, input_cost_usd=0.00278775, cached_input_cost_usd=0.00019295999999999997, output_cost_usd=0.0005285, total_cost_usd=0.004314935)

> 🛟 **Executable fallback below.** The next code cell is intentionally folded. In JupyterLab, click its `…` control (or the blue input gutter at the left) to reveal it, then run the cell.


In [ ]:
#@title Fallback: prove `F4_derivative` { display-mode: "form" }
if f4_derivative.completed:
    print("F4_derivative is already closed.")
else:
    f4_derivative.reset()
    _outputs = f4_derivative.run_script([
        "unfold F4, A4, sech, tanh_exp.",
        "auto_derive.",
        "repeat split.",
        "apply sech_denominator_nonzero.",
        "apply sech_denominator_nonzero.",
        "simpl.",
        "replace (100 * x + - (40)) with (100 * x - 40) by ring.",
        "replace (2 * (100 * x - 40)) with ((100 * x - 40) + (100 * x - 40)) by ring.",
        "rewrite exp_plus.",
        "field.",
        "nra.",
    ])
    for _out in _outputs:
        if not _out.get("ok"):
            raise RuntimeError(_out)
    _qed = f4_derivative.qed()
    if not _qed.get("ok"):
        raise RuntimeError(_qed)
    print(_qed)


## 7. `F6_derivative`

The sixth-power proof is again the same pattern, with more algebra and more denominator side conditions. The selected formal ingredients remain the denominator lemma and `exp_plus`.

Now we pass two nearby examples through `extra_context`: the successful `F2_derivative` and `F4_derivative` proofs. The point is to separate formal ingredients from examples of proof style.


In [35]:
f6_derivative = doc_analytic_integral.ensure_theorem("""Lemma F6_derivative (x : R) :
  is_derive F6 x ((sech (1000 * x - 600)) ^ 6).""")

Safety net (with the correct context)


In [36]:
set_hits(
    selected_hits,
    expected_hit(retriever, "auto_derive", library="Coquelicot", kind="ltac"),
    *technical_derivative_hits,
)


['auto_derive', 'sech_denominator_nonzero', 'exp_plus']


[{'content': 'Ltac auto_derive :=\n  match goal with\n  | |- is_derive ?f ?v ?l =>\n    auto_derive_fun f ;\n    let H := fresh "H" in\n    intro H ;\n    refine (@eq_ind R _ (is_derive f v) (H v _) l _) ;\n    clear H\n  | |- ex_derive ?f ?v =>\n    eexists ;\n    auto_derive_fun f ;\n    let H := fresh "H" in\n    intro H ;\n    apply (H v) ;\n    clear H\n  | |- derivable_pt_lim ?f ?v ?l =>\n    apply is_derive_Reals ;\n    auto_derive\n  | |- derivable_pt ?f ?v =>\n    apply ex_derive_Reals_0 ;\n    auto_derive\n  end.',
  'data_uid': 'auto_derive',
  'depth': 0,
  'docstring': 'A tactic that automatically establishes the differentiability of real-valued functions or computes their derivatives at specified points by processing goals concerning derivative existence, derivative values, or pointwise derivability. The tactic applies a reflective algorithm that symbolically differentiates expressions and reduces proof obligations to verification of domain conditions including non-zero d

In [37]:
f6_example_context = """
Here are nearby successful derivative proofs. Use them as examples of structure,
while adapting the details to the current goal.

```coq
Lemma F2_derivative (x : R) :
  is_derive F2 x ((sech (10 * x - 2)) ^ 2).
Proof.
  unfold F2, A2, sech, tanh_exp.
  auto_derive.
  apply sech_denominator_nonzero.
  simpl.
  replace (10 * x + - (2)) with (10 * x - 2) by ring.
  replace (2 * (10 * x - 2)) with ((10 * x - 2) + (10 * x - 2)) by ring.
  rewrite exp_plus.
  field.
  nra.
Qed.
```
"""

result = remote_llm.prove(
    f6_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f6_derivative.run_tac,
        "reverse": f6_derivative.reverse,
    },
    max_tool_calls=20,
    extra_context=f6_example_context,
)
print(result)
show_usage("F6_derivative", result)


[llm] requesting tool...
[llm:queue] running | queued=0 | running=6 | attempts=1
[llm:queue] succeeded | queued=0 | running=4 | attempts=1
[llm] response received (428 chars) | input=3173, output=175, cost=$0.000563 | queue wait=0.0s, attempts=1, retries=0
[tool] finish: {'action': 'finish', 'script': 'unfold F6, A6, sech, tanh_exp.\nauto_derive.\nsplit.\napply (sech_denominator_nonzero (2 * (1000 * x + - (600)))).\nsplit.\napply (sech_denominator_nonzero (2 * (1000 * x + - (600)))).\nlra.\nreplace (2 * (1000 * x + - (600))) with ((1000 * x - 600) + (1000 * x - 600)) by ring.\nreplace (2 * (1000 * x - 600)) with ((1000 * x - 600) + (1000 * x - 600)) by ring.\nrewrite exp_plus.\nfield; nra.'}
[script]
unfold F6, A6, sech, tanh_exp.
auto_derive.
split.
apply (sech_denominator_nonzero (2 * (1000 * x + - (600)))).
split.
apply (sech_denominator_nonzero (2 * (1000 * x + - (600)))).
lra.
replace (2 * (1000 * x + - (600))) with ((1000 * x - 600) + (1000 * x - 600)) by ring.
replace (2 * (1000

LLMUsage(model='z-ai/glm-5.3-flash', input_tokens=26419, output_tokens=3575, total_tokens=29994, cache_tokens=4224, uncached_input_tokens=22195, input_cost_usd=0.001664625, cached_input_cost_usd=6.336e-05, output_cost_usd=0.00089375, total_cost_usd=0.003694035)

> 🛟 **Executable fallback below.** The next code cell is intentionally folded. In JupyterLab, click its `…` control (or the blue input gutter at the left) to reveal it, then run the cell.


In [ ]:
#@title Fallback: prove `F6_derivative` { display-mode: "form" }
if f6_derivative.completed:
    print("F6_derivative is already closed.")
else:
    f6_derivative.reset()
    _outputs = f6_derivative.run_script([
        "unfold F6, A6, sech, tanh_exp.",
        "auto_derive.",
        "repeat split.",
        "apply sech_denominator_nonzero.",
        "apply sech_denominator_nonzero.",
        "apply sech_denominator_nonzero.",
        "trivial.",
        "simpl.",
        "replace (1000 * x + - (600)) with (1000 * x - 600) by ring.",
        "replace (2 * (1000 * x - 600)) with ((1000 * x - 600) + (1000 * x - 600)) by ring.",
        "rewrite exp_plus.",
        "field.",
        "nra.",
    ])
    for _out in _outputs:
        if not _out.get("ok"):
            raise RuntimeError(_out)
    _qed = f6_derivative.qed()
    if not _qed.get("ok"):
        raise RuntimeError(_qed)
    print(_qed)


In [38]:
selected_hits.append(f6_derivative.as_retrieval_hit())

## 8. Combining the derivative lemmas

Now we need to prove that the derivative of the sum `F2 + F4 + F6` is the sum of the derivatives. The formal ingredient to retrieve is the Coquelicot rule for the derivative of a sum.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint — click to reveal</strong></summary>

Try a query like: `derivative of the sum of two real functions`.

</details>

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hit — click to reveal</strong></summary>

Name: `is_derive_plus`

Signature:

```coq
is_derive_plus :
  forall (f g : R -> R) (x df dg : R),
    is_derive f x df ->
    is_derive g x dg ->
    is_derive (fun y => f y + g y) x (df + dg)
```

</details>


In [39]:
set_hits(
    selected_hits,
    f2_derivative.as_retrieval_hit(),
    f4_derivative.as_retrieval_hit(),
    f6_derivative.as_retrieval_hit(),
)

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="theorem",
    default_k=8,
)
retrieval_explorer.display()


['F2_derivative', 'F4_derivative', 'F6_derivative']


In [40]:
f_derivative = doc_analytic_integral.ensure_theorem("""Lemma F_derivative (x : R) :
  is_derive F x (f x).""")


Safety net: load the sum-derivative rule together with the three derivative lemmas proved above.


In [41]:
set_hits(
    selected_hits,
    f2_derivative.as_retrieval_hit(),
    f4_derivative.as_retrieval_hit(),
    f6_derivative.as_retrieval_hit(),
    expected_hit(retriever, "is_derive_plus", library="Coquelicot", kind="theorem"),
)


['F2_derivative', 'F4_derivative', 'F6_derivative', 'is_derive_plus']


[{'uid': 'local:F2_derivative',
  'name': 'F2_derivative',
  'kind': 'local_theorem',
  'library': 'Current document',
  'source': 'current Rocq session',
  'statement': 'Lemma F2_derivative (x : R) :\n  is_derive F2 x ((sech(10*x - 2))^2).',
  'docstring': 'Lemma F2_derivative (x : R) :\n  is_derive F2 x ((sech(10*x - 2))^2).\nProof.\n  unfold F2, A2, sech, tanh_exp.\n  auto_derive.\n  apply (sech_denominator_nonzero (2 * (10 * x - 2))).\n  replace (2 * (10 * x + - (2))) with ((10 * x - 2) + (10 * x - 2)) by ring.\n  replace (2 * (10 * x - 2)) with ((10 * x - 2) + (10 * x - 2)) by ring.\n  rewrite exp_plus.\n  field; nra.\nQed.\n',
  'content': 'Lemma F2_derivative (x : R) :\n  is_derive F2 x ((sech(10*x - 2))^2).\nProof.\n  unfold F2, A2, sech, tanh_exp.\n  auto_derive.\n  apply (sech_denominator_nonzero (2 * (10 * x - 2))).\n  replace (2 * (10 * x + - (2))) with ((10 * x - 2) + (10 * x - 2)) by ring.\n  replace (2 * (10 * x - 2)) with ((10 * x - 2) + (10 * x - 2)) by ring.\n  rewrit

In [42]:
result = remote_llm.prove(
    f_derivative,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": f_derivative.run_tac,
        "reverse": f_derivative.reverse,
    },
    max_tool_calls=20,
    extra_context="""Use the derivative rule for sums together with the derivative lemmas already proved.""",
)
print(result)
show_usage("F_derivative", result)


[llm] requesting tool...
[llm:queue] running | queued=0 | running=5 | attempts=1
[llm:queue] succeeded | queued=0 | running=3 | attempts=1
[llm] response received (158 chars) | input=3649, output=65, cost=$0.000580 | queue wait=0.0s, attempts=1, retries=0
[tool] finish: {'action': 'finish', 'script': 'unfold F, f.\napply is_derive_plus.\napply is_derive_plus.\napply F2_derivative.\napply F4_derivative.\napply F6_derivative.'}
[script]
unfold F, f.
apply is_derive_plus.
apply is_derive_plus.
apply F2_derivative.
apply F4_derivative.
apply F6_derivative.
[rocq] unfold F, f.
[rocq] ok, goals=1
[rocq] apply is_derive_plus.
[rocq] failed
[rocq:error] (-32003, 'Coq: In environment\nx : R\nUnable to unify\n "is_linear (fun y : ?K => scal y (plus ?M2177 ?M2178)) /\\\n (forall x0 : AbsRing_NormedModule ?K,\n is_filter_lim (locally ?M2176) x0 ->\n is_domin (locally ?M2176) (fun y : AbsRing_NormedModule ?K => minus y x0)\n (fun y : AbsRing_NormedModule ?K =>\n minus\n (minus ((fun x1 : ?K => plus

LLMUsage(model='z-ai/glm-5.3-flash', input_tokens=18192, output_tokens=1199, total_tokens=19391, cache_tokens=0, uncached_input_tokens=18192, input_cost_usd=0.0013643999999999998, cached_input_cost_usd=0.0, output_cost_usd=0.00029975, total_cost_usd=0.0033283)

> 🛟 **Executable fallback below.** The next code cell is intentionally folded. In JupyterLab, click its `…` control (or the blue input gutter at the left) to reveal it, then run the cell.


In [ ]:
#@title Fallback: prove `F_derivative` { display-mode: "form" }
if f_derivative.completed:
    print("F_derivative is already closed.")
else:
    f_derivative.reset()
    _outputs = f_derivative.run_script([
        "unfold F, f.",
        "apply is_derive_plus with (f := fun x0 => ((F2 x0) + (F4 x0))) (g := F6).",
        "apply is_derive_plus with (f := F2) (g := F4).",
        "apply F2_derivative.",
        "apply F4_derivative.",
        "apply F6_derivative.",
    ])
    for _out in _outputs:
        if not _out.get("ok"):
            raise RuntimeError(_out)
    _qed = f_derivative.qed()
    if not _qed.get("ok"):
        raise RuntimeError(_qed)
    print(_qed)


## 9. What the integral theorem asks for

The closed-form theorem uses the fundamental theorem of calculus: `F` is an antiderivative of `f`, so the integral should be `F 1 - F 0`.


First, search for the theorem that turns an integral of a derivative as a difference between its bounds.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint — click to reveal</strong></summary>

Try a query like: `integral of the derivative is the difference between the bounds`.

</details>

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hit — click to reveal</strong></summary>

Name: `is_RInt_derive`

```coq
is_RInt_derive :
  forall (f g : R -> R) (a b : R),
    (forall x, Rmin a b <= x <= Rmax a b -> is_derive f x (g x)) ->
    (forall x, Rmin a b <= x <= Rmax a b -> continuous g x) ->
    is_RInt g a b (f b - f a)
```

</details>


In [43]:
integral_hits = []

retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=integral_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="theorem",
    default_k=8,
)
retrieval_explorer.display()


Coquelicot first proves integrals in the form `is_RInt f a b v`, meaning that `v` is the value of the integral. Here `is_RInt_derive` proves that `F 1 - F 0` has this property, while our goal is written with the function `RInt f 0 1`. We therefore need a lemma to turn the `is_RInt` statement into the equality about `RInt`.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint for the is_RInt/RInt bridge — click to reveal</strong></summary>

Try a query like: `is_RInt is equal to RInt`.

</details>

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hit — click to reveal</strong></summary>

Name: `is_RInt_unique`

Signature:

```coq
is_RInt_unique : forall f a b If, is_RInt f a b If -> RInt f a b = If
```

</details>


In [44]:
retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=integral_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="theorem",
    default_k=8,
)
retrieval_explorer.display()


The final theorem now has two obligations. The antiderivative obligation is handled by `F_derivative`; the missing part is continuity of `f` on the interval.


## 10. The missing continuity lemma

We prove `f_continuous` directly. The main theorem to find is the one saying that differentiability (that we prove many times) implies continuity; the derivative obligation will be solved inside the same proof with `auto_derive` and `sech_denominator_nonzero`.


Start by searching for the theorem saying that a differentiable real function is continuous.

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>💡 Hint for the continuity lemma — click to reveal</strong></summary>

Try a query like: `differentiable function is continuous`.

</details>

<details style="margin:0.75rem 0; padding:0.55rem 0.75rem; border:1px solid #79a7e3; border-radius:0.4rem; background:#f5f9ff; color:#1f2328;">
<summary><strong>✅ Expected retrieval hit for the continuity lemma — click to reveal</strong></summary>

Name: `ex_derive_continuous`

Signature:

```coq
ex_derive_continuous :
  forall (f : R -> R) (x : R), ex_derive f x -> continuous f x
```

</details>


In [45]:
selected_hits.clear()
retrieval_explorer = RetrievalExplorer(
    retriever,
    selected_hits=selected_hits,
    default_query="",
    default_library="Coquelicot",
    default_kind="theorem",
    default_k=8,
)
retrieval_explorer.display()

In [46]:
f_continuous = doc_analytic_integral.ensure_theorem("""Lemma f_continuous (x : R) :
  continuous f x.""")


Safety net: add the derivative tactic and the denominator lemma by hand, so the call has everything needed for the differentiability subgoal.


In [47]:
set_hits(
    selected_hits,
    expected_hit(retriever, "ex_derive_continuous", library="Coquelicot", kind="theorem"),
    expected_hit(retriever, "auto_derive", library="Coquelicot", kind="ltac"),
    sech_denominator_nonzero.as_retrieval_hit(),
)


['ex_derive_continuous', 'auto_derive', 'sech_denominator_nonzero']


[{'content': 'Lemma ex_derive_continuous (f : K -> V) (x : K) :\n  ex_derive f x -> continuous f x.',
  'data_uid': 'ex_derive_continuous',
  'depth': 0,
  'docstring': 'This lemma states that if a function from a scalar absolute ring to a normed module has a derivative at a point, then the function is continuous at that point.',
  'env_id': 'coq-coquelicot',
  'kind': 'theorem',
  'library': 'Coquelicot',
  'name': 'ex_derive_continuous',
  'preorder_index': 858,
  'range': [31533, 31617],
  'source': 'Coquelicot/Derive.v',
  'source_id': '56341e8d60fe128c31fbaf9d0148efbc42991cd9e918a8f4c80a0d63e7dffe8d',
  'uid': 'fc7713d11cd6c0c14b7eed26985b4ebf7fbe1646',
  'score': 1.0},
 {'content': 'Ltac auto_derive :=\n  match goal with\n  | |- is_derive ?f ?v ?l =>\n    auto_derive_fun f ;\n    let H := fresh "H" in\n    intro H ;\n    refine (@eq_ind R _ (is_derive f v) (H v _) l _) ;\n    clear H\n  | |- ex_derive ?f ?v =>\n    eexists ;\n    auto_derive_fun f ;\n    let H := fresh "H" in\n  

In [48]:
result = remote_llm.prove(
    f_continuous,
    selected_hits=selected_hits,
    extra_context=(
        "Prove continuity directly, without using an external differentiability lemma. "
        "First apply ex_derive_continuous, then unfold f and sech, use auto_derive, "
        "and solve the remaining denominator goals with sech_denominator_nonzero."
    ),
    verbose=True,
    tools={
        "run_tac": f_continuous.run_tac,
        "reverse": f_continuous.reverse,
    },
    max_tool_calls=16,
)
print(result)
show_usage("f_continuous", result)


[llm] requesting tool...
[llm:queue] running | queued=0 | running=5 | attempts=1
[llm:queue] succeeded | queued=0 | running=3 | attempts=1
[llm] response received (147 chars) | input=3327, output=318, cost=$0.000658 | queue wait=0.0s, attempts=1, retries=0
[tool] run_tac: {'action': 'run_tac', 'tactic': 'apply ex_derive_continuous.', 'allow_new_goals': False, 'reason': 'reduces continuity to existence of derivative'}
[rocq] apply ex_derive_continuous.
[rocq] failed, goals=1
[rocq:error] (-32003, 'Coq: In environment\nx : R\nP : ?V -> Prop\nUnable to unify\n "@ex posreal\n (fun eps : posreal =>\n forall (y : UniformSpace.sort (NormedModule.UniformSpace ?K ?V))\n (_ : @ball (NormedModule.UniformSpace ?K ?V) \n (?M2185 ?M2186) (pos eps) y), P y)"\nwith\n "@ex posreal\n (fun eps : posreal =>\n forall (y : UniformSpace.sort R_UniformSpace)\n (_ : @ball R_UniformSpace (f x) (pos eps) y), \n P y)".')
[llm] requesting tool...
[llm:queue] running | queued=0 | running=3 | attempts=1
[llm:queue] 

LLMUsage(model='z-ai/glm-5.3-flash', input_tokens=19450, output_tokens=1937, total_tokens=21387, cache_tokens=1408, uncached_input_tokens=18042, input_cost_usd=0.0013531499999999998, cached_input_cost_usd=2.1119999999999998e-05, output_cost_usd=0.00048425, total_cost_usd=0.003423345)

> 🛟 **Executable fallback below.** The next code cell is intentionally folded. In JupyterLab, click its `…` control (or the blue input gutter at the left) to reveal it, then run the cell.


In [ ]:
#@title Fallback: prove `f_continuous` { display-mode: "form" }
if f_continuous.completed:
    print("f_continuous is already closed.")
else:
    f_continuous.reset()
    _outputs = f_continuous.run_script([
        "apply (ex_derive_continuous f x).",
        "unfold f, sech.",
        "auto_derive.",
        "repeat split.",
        "all: apply sech_denominator_nonzero.",
    ])
    for _out in _outputs:
        if not _out.get("ok"):
            raise RuntimeError(_out)
    _qed = f_continuous.qed()
    if not _qed.get("ok"):
        raise RuntimeError(_qed)
    print(_qed)


## 11. Closed form of the integral

We can now return to the theorem that motivated the continuity detour. The final context contains:

- the integration facts collected in Section 9;
- the antiderivative fact `F_derivative`;
- the continuity fact `f_continuous`.


In [49]:
set_hits(
    selected_hits,
    expected_hit(retriever, "is_RInt_derive", library="Coquelicot", kind="theorem"),
    expected_hit(retriever, "is_RInt_unique", library="Coquelicot", kind="theorem"),
    f_derivative.as_retrieval_hit(),
    f_continuous.as_retrieval_hit(),
)


['is_RInt_derive', 'is_RInt_unique', 'F_derivative', 'f_continuous']


[{'content': 'Lemma is_RInt_derive (f df : R -> V) (a b : R) :\n  (forall x : R, Rmin a b <= x <= Rmax a b -> is_derive f x (df x)) ->\n  (forall x : R, Rmin a b <= x <= Rmax a b -> continuous df x) ->\n  is_RInt df a b (minus (f b) (f a)).',
  'data_uid': 'is_RInt_derive',
  'depth': 0,
  'docstring': 'This lemma states that the Riemann integral of the derivative of a vector-valued function over a closed interval is equal to the difference between the values of the function at the upper and lower bounds of that interval, provided the function is differentiable everywhere on the interval and its derivative is continuous there.',
  'env_id': 'coq-coquelicot',
  'kind': 'theorem',
  'library': 'Coquelicot',
  'name': 'is_RInt_derive',
  'preorder_index': 419,
  'range': [16847, 17070],
  'source': 'Coquelicot/RInt_analysis.v',
  'source_id': 'f6ea192c73cdd186e13697b624047242ce62d0fe6d9e1a474e07db8500fbcd3d',
  'uid': '1595bb05404bfac395ec0cc003497705a08a97b9',
  'score': 1.0},
 {'content

In [50]:
I_closed_form_correct = doc_analytic_integral.ensure_theorem("""Theorem I_closed_form_correct :
  I = I_closed_form.""")

result = remote_llm.prove(
    I_closed_form_correct,
    selected_hits=selected_hits,
    verbose=True,
    tools={
        "run_tac": I_closed_form_correct.run_tac,
        "reverse": I_closed_form_correct.reverse,
    },
    max_tool_calls=24,
)
print(result)
show_usage("I_closed_form_correct", result)


[llm] requesting tool...
[llm:queue] running | queued=0 | running=5 | attempts=1
[llm:queue] succeeded | queued=0 | running=3 | attempts=1
[llm] response received (177 chars) | input=3442, output=232, cost=$0.000632 | queue wait=0.0s, attempts=1, retries=0
[tool] finish: {'action': 'finish', 'script': 'unfold I, I_closed_form.\napply is_RInt_unique.\napply (is_RInt_derive F f 0 1).\nintros x _. apply F_derivative.\nintros x _. apply f_continuous.'}
[script]
unfold I, I_closed_form.
apply is_RInt_unique.
apply (is_RInt_derive F f 0 1).
intros x _. apply F_derivative.
intros x _. apply f_continuous.
[rocq] unfold I, I_closed_form.
[rocq] ok, goals=1
[rocq] apply is_RInt_unique.
[rocq] ok, goals=1
[rocq] apply (is_RInt_derive F f 0 1).
[rocq] ok, goals=2
[rocq] intros x _.
[rocq] ok, goals=2
[rocq] apply F_derivative.
[rocq] ok, goals=1
[rocq] intros x _.
[rocq] ok, goals=1
[rocq] apply f_continuous.
[rocq] ok, goals=0
[rocq] Qed ok
ProofResult(ok)

usage: input=3442, output=232, cost=$0.

LLMUsage(model='z-ai/glm-5.3-flash', input_tokens=3442, output_tokens=232, total_tokens=3674, cache_tokens=0, uncached_input_tokens=3442, input_cost_usd=0.00025814999999999996, cached_input_cost_usd=0.0, output_cost_usd=5.8e-05, total_cost_usd=0.0006323)

> 🛟 **Executable fallback below.** The next code cell is intentionally folded. In JupyterLab, click its `…` control (or the blue input gutter at the left) to reveal it, then run the cell.


In [ ]:
#@title Fallback: prove `I_closed_form_correct` { display-mode: "form" }
if I_closed_form_correct.completed:
    print("I_closed_form_correct is already closed.")
else:
    I_closed_form_correct.reset()
    _outputs = I_closed_form_correct.run_script([
        "unfold I, I_closed_form.",
        "apply is_RInt_unique.",
        "apply (is_RInt_derive F f 0 1).",
        "intros x _. apply F_derivative.",
        "intros x _. apply f_continuous.",
    ])
    for _out in _outputs:
        if not _out.get("ok"):
            raise RuntimeError(_out)
    _qed = I_closed_form_correct.qed()
    if not _qed.get("ok"):
        raise RuntimeError(_qed)
    print(_qed)


The formal result is now:

```coq
I = F 1 - F 0
```

So the numerical dispute can be checked again by evaluating a stable version of the closed form in ordinary Python. This final Python calculation is not the proof; the proof is the Rocq theorem above.


In [51]:
import math


def A2_num(u):
    t = math.tanh(u)
    return t


def A4_num(u):
    t = math.tanh(u)
    return t - (1 / 3) * t**3


def A6_num(u):
    t = math.tanh(u)
    return t - (2 / 3) * t**3 + (1 / 5) * t**5


def F_num(x):
    return (
        A2_num(10 * x - 2) / 10
        + A4_num(100 * x - 40) / 100
        + A6_num(1000 * x - 600) / 1000
    )

F_num(1) - F_num(0)


0.2108027355005493

The value is about `0.2108027355005493`, which lies inside the certified Rocq interval.


## 12. Source generated by the analytic document

For debugging or comparison with `integral.v`, print the source accumulated in the analytic document.


In [52]:
print(doc_analytic_integral.source())


From Coq Require Import Reals Lra Psatz.

From Coquelicot Require Import Coquelicot.

From Interval Require Import Tactic Plot.

Definition sech (u : R) : R :=
  2 * exp (u) / (exp (2 * u) + 1).

Definition f (x : R) : R :=
    (sech (10 * x - 2))^2
  + (sech (100 * x - 40))^4
  + (sech (1000 * x - 600))^6.

Definition I : R := RInt f 0 1.

Definition tanh_exp (u : R) : R :=
  (exp (2 * u) - 1) / (exp (2 * u) + 1).

Definition A2 (u : R) : R :=
  tanh_exp u.

Definition A4 (u : R) : R :=
  tanh_exp u - (/ 3) * (tanh_exp u)^3.

Definition A6 (u : R) : R :=
  tanh_exp u - (2 / 3) * (tanh_exp u)^3 + (/ 5) * (tanh_exp u)^5.

Definition F2 (x : R) : R :=
  A2 (10 * x - 2) / 10.

Definition F4 (x : R) : R :=
  A4 (100 * x - 40) / 100.

Definition F6 (x : R) : R :=
  A6 (1000 * x - 600) / 1000.

Definition F (x : R) : R :=
  F2 x + F4 x + F6 x.

Definition I_closed_form : R :=
  F 1 - F 0.

Lemma sech_denominator_nonzero (u : R) :
  exp u + 1 <> 0.
Proof.
  apply Rgt_not_eq.
  apply Rplus_lt_